In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import LearningRateScheduler
import matplotlib.pyplot as plt
import cv2

: 

In [ ]:
print(f"X_test shape: {X_test.shape}")
print(f"Y_test shape: {Y_test.shape}")

In [ ]:
Y_test = np.argmax(Y_test, axis=-1)  # Convert one-hot to label encoding

In [ ]:
# Initialize list to store test accuracies
test_accuracies = []

# Loop through each trained model
for j in range(nets):
    test_loss, test_acc = model[j].evaluate(X_test, Y_test, verbose=0)
    test_accuracies.append(test_acc)
    print(f"CNN {j+1}: Test Accuracy = {test_acc:.5f}")

# Compute the average accuracy across all models
average_test_accuracy = np.mean(test_accuracies)
print(f"Average Test Accuracy: {average_test_accuracy:.5f}")

In [ ]:
# Get predictions from all models
predictions = np.zeros((X_test.shape[0], 10))  # Assuming 10 classes

for j in range(nets):  # nets = 15
    predictions += model[j].predict(X_test, verbose=0)  # Sum the predictions

# Average the predictions
predictions /= nets  # Take the mean across all models

# Convert probabilities to class labels
final_predictions = np.argmax(predictions, axis=1)

# Ensure Y_test is in the correct format (integer labels)
Y_test_labels = np.argmax(Y_test, axis=1) if Y_test.shape[1] > 1 else Y_test

# Compute accuracy
ensemble_accuracy = np.mean(final_predictions == Y_test_labels)
print(f"Ensemble Model Accuracy: {ensemble_accuracy:.5f}")

In [ ]:
# Generate ensemble predictions
results = np.zeros((X_test.shape[0], 10))
for j in range(nets):
    results += model[j].predict(X_test)

# Convert probabilities to class labels
final_predictions = np.argmax(results, axis=1)

# Plot the first 40 test images with their predictions
plt.figure(figsize=(15, 6))
for i in range(40):
    plt.subplot(4, 10, i+1)
    plt.imshow(X_test[i].reshape((28, 28)), cmap='gray')  # Ensure grayscale image
    plt.title(f"Pred: {final_predictions[i]}", fontsize=10, y=0.9)
    plt.axis('off')

plt.subplots_adjust(wspace=0.3, hspace=-0.1)
plt.show()

In [ ]:
import tensorflow as tf

# Save any one of your best trained models
best_model = model[0]  # Choose your best performing CNN
best_model.save("./digit_recognizer.h5")

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
model = load_model('C:/Users/user/please/model/training/digit_recognizer.h5')

# Function to preprocess the image
def preprocess_image(img_path, target_size):
    # Load the image using OpenCV
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)  # Read as grayscale
    img = cv2.resize(img, target_size)  # Resize to match model input size
    img = img.astype('float32') / 255.0  # Normalize to [0, 1]
    img = np.expand_dims(img, axis=-1)  # Add channel dimension
    img = np.expand_dims(img, axis=0)  # Add batch dimension
    return img

# Path to your handwritten image
image_path = 'C:/Users/user/please/model/training/3hetrhjetdryh.jpg'  # Replace with your image path

# Preprocess the image
target_size = (28, 28)  # Replace with your model's input size
processed_image = preprocess_image(image_path, target_size)

# Make a prediction
prediction = model.predict(processed_image)
predicted_class = np.argmax(prediction, axis=1)

# Display the result
print(f"Predicted Class: {predicted_class[0]}")
print(f"Prediction Probabilities: {prediction}")

# Optional: Display the image
img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
plt.imshow(img, cmap='gray')
plt.title(f"Predicted Class: {predicted_class[0]}")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Assuming you have a history object from model.fit()
history = model.fit(
    train,  # Your training data
    epochs=20,        # Number of epochs
    validation_data=validation_generator  # Your validation data
)

# Plot training & validation loss values
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Plot training & validation accuracy values
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()